#  Demystifying LLMs — The Transformer Block

This notebook walks through **every layer inside a Transformer block**, step by step, with real numbers.
No black boxes. We build it from scratch using only `numpy` and `torch`.

**What we'll cover:**
1. Input — tokenization to vectors
2. Multi-Head Attention (Q, K, V)
3. Add & Norm (residual connection + layer norm)
4. Feed Forward Network (FFN)
5. Add & Norm again
6. Putting it all together in one block

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Reproducibility
torch.manual_seed(42)

print('Libraries loaded. Let\'s build a Transformer block from scratch.')

## Step 1 — The Input

In a real LLM, your sentence goes through a tokenizer and an embedding layer before reaching the Transformer block.
Here we simulate that: **3 tokens, each represented as a vector of 8 dimensions**.

Think of it as: `"the cat sat"` → 3 token vectors.

> In GPT-4, each token vector has **12,288 dimensions**. We use 8 to keep the math visible.

In [ ]:
# Hyperparameters
seq_len   = 3    # number of tokens: "the", "cat", "sat"
d_model   = 8    # embedding dimension (tiny, for readability)
num_heads = 2    # attention heads
d_ff      = 32   # FFN inner dimension (usually 4 × d_model)
d_head    = d_model // num_heads  # dimension per head = 4

# Simulated input: 3 tokens, each with 8 dimensions
# Shape: (seq_len, d_model)
x = torch.randn(seq_len, d_model)

print(f'Input shape: {x.shape}  → ({seq_len} tokens, {d_model} dimensions each)')
print(f'\nToken vectors:')
print(f'  "the" → {x[0].detach().numpy().round(3)}')
print(f'  "cat" → {x[1].detach().numpy().round(3)}')
print(f'  "sat" → {x[2].detach().numpy().round(3)}')

## Step 2 — Multi-Head Attention (Q, K, V)

This is where tokens **talk to each other**.

Each token produces three vectors:
- **Q (Query)** → *"What am I looking for?"*
- **K (Key)**   → *"What do I advertise to others?"*
- **V (Value)** → *"What information do I actually carry?"*

The attention score between token A and token B is:
```
score(A, B) = softmax( Q_A · K_B / √d_head )
```
Then each token updates itself by blending the V vectors, weighted by those scores.

We run this in **parallel across multiple heads** — each head learns to look for different relationships.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_head    = d_model // num_heads

        # One linear layer each for Q, K, V — then one to merge heads back
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # Query matrix
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # Key matrix
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # Value matrix
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # Output projection

    def forward(self, x, verbose=False):
        seq_len, d_model = x.shape

        # 1. Project input into Q, K, V
        Q = self.W_q(x)  # (seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. Split into multiple heads
        #    reshape: (seq_len, num_heads, d_head) → (num_heads, seq_len, d_head)
        Q = Q.view(seq_len, self.num_heads, self.d_head).transpose(0, 1)
        K = K.view(seq_len, self.num_heads, self.d_head).transpose(0, 1)
        V = V.view(seq_len, self.num_heads, self.d_head).transpose(0, 1)

        # 3. Attention scores: Q · Kᵀ / √d_head
        scale  = math.sqrt(self.d_head)
        scores = torch.bmm(Q, K.transpose(1, 2)) / scale  # (num_heads, seq_len, seq_len)

        # 4. Softmax → attention weights (probabilities, sum to 1 per row)
        weights = F.softmax(scores, dim=-1)

        if verbose:
            print('Attention weights (head 0) — rows=query token, cols=key token:')
            labels = ['"the"', '"cat"', '"sat"']
            header = '          ' + '  '.join(f'{l:>7}' for l in labels)
            print(header)
            for i, row in enumerate(weights[0]):
                print(f'  {labels[i]:>7} ', end='')
                for w in row:
                    bar = '█' * int(w.item() * 20)
                    print(f'  {w.item():.3f} {bar:<20}', end='')
                print()
            print()

        # 5. Weighted sum of V vectors
        attended = torch.bmm(weights, V)  # (num_heads, seq_len, d_head)

        # 6. Concatenate heads and project back to d_model
        attended = attended.transpose(0, 1).contiguous().view(seq_len, d_model)
        output   = self.W_o(attended)

        return output, weights


mha = MultiHeadAttention(d_model, num_heads)
attn_output, attn_weights = mha(x, verbose=True)

print(f'Attention output shape: {attn_output.shape}  → same as input (tokens × dimensions)')
print(f'\nEach token now carries context from all others.')

## Step 3 — Add & Norm (first one)

Two things happen here:

1. **Residual connection (Add):** We add the original input `x` back to the attention output.
   This is the shortcut path — it lets gradients flow cleanly during training, even across 96 stacked blocks.
   
   ```python
   x = x + attention_output
   ```

2. **Layer Normalization (Norm):** Rescales the values so they don't drift out of control
   as we pass through many layers.

> **Analogy:** Imagine you're in a telephone game (Chinese whispers). After each round, someone recaps the original message so it doesn't drift completely. That's the residual connection.

In [ ]:
layer_norm_1 = nn.LayerNorm(d_model)

# Add (residual) + Norm
x_after_attn = layer_norm_1(x + attn_output)

print('Before Add & Norm (original input, first token):')
print(f'  {x[0].detach().numpy().round(3)}')
print(f'  mean={x[0].mean().item():.3f}  std={x[0].std().item():.3f}')

print('\nAfter Add & Norm (first token):')
print(f'  {x_after_attn[0].detach().numpy().round(3)}')
print(f'  mean={x_after_attn[0].mean().item():.3f}  std={x_after_attn[0].std().item():.3f}')
print('\n→ Values are normalized. Mean ≈ 0, Std ≈ 1. Stable for the next layer.')

## Step 4 — Feed Forward Network (FFN)

Now each token goes **solo**. No more looking at neighbors.

The FFN is a tiny two-layer neural network applied **independently to each token vector**:

```
FFN(x) = Linear₂( ReLU( Linear₁(x) ) )
```

- **Linear₁:** Expands from `d_model=8` → `d_ff=32` (4× wider)
- **ReLU:** Kills all negative values → zeroes out useless combinations
- **Linear₂:** Compresses back from `d_ff=32` → `d_model=8`

This is where a lot of the model's **factual knowledge** lives — stored in the weights of these linear layers.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)   # expand
        self.linear2 = nn.Linear(d_ff, d_model)   # compress

    def forward(self, x, verbose=False):
        expanded = F.relu(self.linear1(x))         # (seq_len, d_ff)
        output   = self.linear2(expanded)           # (seq_len, d_model)

        if verbose:
            alive = (expanded > 0).float().mean().item()
            print(f'After Linear₁ + ReLU:')
            print(f'  Shape: {expanded.shape}  → expanded from {x.shape[1]} to {expanded.shape[1]} dims')
            print(f'  Neurons alive after ReLU: {alive:.1%}  (the rest were zeroed out)')
            print(f'\nAfter Linear₂ (compression back):')
            print(f'  Shape: {output.shape}  → back to {output.shape[1]} dims')

        return output


ffn = FeedForward(d_model, d_ff)
ffn_output = ffn(x_after_attn, verbose=True)

print(f'\n→ Each token processed independently. Same FFN weights, applied like a map() over the sequence.')

## Step 5 — Add & Norm (second one)

Same as before — residual connection + layer norm, but now after the FFN.

```python
x = LayerNorm( x + ffn_output )
```

The block always ends with stable, normalized vectors — ready to be the input of the **next block**.

In [ ]:
layer_norm_2 = nn.LayerNorm(d_model)

# Add (residual) + Norm
x_out = layer_norm_2(x_after_attn + ffn_output)

print('Final output of the Transformer block:')
print(f'  Shape: {x_out.shape}  → same as input ({seq_len} tokens × {d_model} dims)')
print()
for i, label in enumerate(['"the"', '"cat"', '"sat"']):
    print(f'  {label:>7} → {x_out[i].detach().numpy().round(3)}')

print()
print('→ These vectors are richer than when they came in.')
print('  Each token now carries meaning shaped by attention, FFN, and two normalizations.')
print('  In GPT-4, this block runs 96 more times before any output is produced.')

## Step 6 — The Full Transformer Block

Let's put it all together in one clean class — the way it actually looks in production code.

In [ ]:
class TransformerBlock(nn.Module):
    """
    One full Transformer block:
      1. Multi-Head Attention
      2. Add & Norm
      3. Feed Forward Network
      4. Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention   = MultiHeadAttention(d_model, num_heads)
        self.ffn         = FeedForward(d_model, d_ff)
        self.norm1       = nn.LayerNorm(d_model)
        self.norm2       = nn.LayerNorm(d_model)

    def forward(self, x):
        # 1. Attention + residual + norm
        attn_out, _ = self.attention(x)
        x = self.norm1(x + attn_out)          # Add & Norm

        # 2. FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)           # Add & Norm

        return x


# --- Run it ---
block  = TransformerBlock(d_model, num_heads, d_ff)
tokens = torch.randn(seq_len, d_model)         # fresh input
output = block(tokens)

print('TransformerBlock — full forward pass')
print(f'  Input  shape: {tokens.shape}')
print(f'  Output shape: {output.shape}')
print()
print('Parameter count per block:')
total = sum(p.numel() for p in block.parameters())
print(f'  {total:,} parameters')
print(f'  GPT-4 has ~96 blocks × ~millions of params each = hundreds of billions total')

## Bonus — Stack N Blocks (like a real LLM)

A real LLM stacks this block N times. Each block refines the token vectors a little more.
Let's stack 4 blocks and watch the vectors evolve.

In [ ]:
N = 4  # number of stacked blocks (GPT-4 uses 96)

stack  = nn.Sequential(*[TransformerBlock(d_model, num_heads, d_ff) for _ in range(N)])
tokens = torch.randn(seq_len, d_model)

print(f'Running {seq_len} tokens through {N} stacked Transformer blocks...\n')
print(f'  Input  (block 0): {tokens[0].detach().numpy().round(3)}')

x = tokens
for i, block in enumerate(stack):
    x = block(x)
    print(f'  Output (block {i+1}): {x[0].detach().numpy().round(3)}')

print()
print('Each pass reshapes the token vectors — refining meaning, resolving ambiguity.')
print('By block 96, "it" in "the animal didn\'t cross because it was tired" is unambiguous.')

---

## Summary

| Layer | What it does | Sees neighbors? |
|---|---|---|
| **Multi-Head Attention** | Tokens look at each other via Q, K, V | ✅ Yes |
| **Add & Norm** | Stabilize + preserve original signal | — |
| **Feed Forward Network** | Each token processes independently | ❌ No |
| **Add & Norm** | Stabilize again before next block | — |

This block is the **entire repeating unit** of every modern LLM — GPT, Llama, Claude, Gemini.
Stack it deep enough, train it on enough data, and magic happens.

> **Next:** How does the model actually *learn* these weights? → Backpropagation, loss functions, and gradient descent. 🧠